## Understanding Attention

In [ ]:
! pip install torch

In [415]:
import numpy as np

### Matrix Multiplication



In [416]:
A = np.random.rand(3, 4) # 3 rows, 4 columns or 3 rows with 4 elements each
B = np.random.rand(3, 4)

In [ ]:
assert A.shape == B.shape
A, B, A*B, np.multiply(A,B) # hadamard product i.e. element-wise 

In [ ]:
# verify
idx1 = np.random.randint(0,3)
idx2 = np.random.randint(0,3)

assert A[idx1][idx2]*B[idx1][idx2] == (A*B)[idx1][idx2], "Incorrect Element-wise Product"
A*B == np.multiply(A,B)

In [ ]:
# Transpose of a matrix
A, A.shape, A.T, A.T.shape

In [ ]:
# Matrix Multiplication
try:
    np.matmul(A,B)
except ValueError as ve:
    print(ve)
# What does this error mean? ((n?,k),(k,m?)->(n?,m?)) 
# n,k are rows, cols of A and k,m should be the rows, cols of B
# Essentially cols of A should be equal to rows of B
try:
    assert len(A[0,:]) == len(B[:, 0]), "A cols is not equal to B rows" # check if number of rows in A is equal to number of columns in B
except AssertionError as ae:
    print(ae)

In [ ]:
# Let's Transpose B and check
assert len(A[0,:]) == len(B.T[:, 0]), "A rows is not equal to B cols" # check if number of rows in A is equal to number of columns in B
A@B.T, np.matmul(A,B.T), np.matmul(A,B.T) == A@B.T

In [ ]:
# dot product
try:
    np.dot(A, B)
except ValueError as ve:
    print(ve)
# We need the same criteria here
np.dot(A,B.T) # why is it the same as matmul?

In [423]:
def show_vector_dot(a, b):
    assert len(a) == len(b), "Vectors must be the same length"

    row = "[" + ", ".join(str(x) for x in a) + "] ×"
    col = ["     " + str(x) for x in b]

    print(row)
    print("[")
    for line in col:
        print(line)
    print("]")

In [ ]:
# dot product is an operation between two vectors, matmul is a composition of dot products
for i in range(len(A[:,0])): # rows
    rowdots = []
    for j in range(len(B[:,0])): # cols
        # print(f"Multiplying {A[i,:]} with {np.array2string(B.T[:,j].reshape(-1, 1), separator=', ')}")
        show_vector_dot(A[i,:], B.T[:,j])
        rowdots += [np.dot(A[i,:], B.T[:,j])]
    print(rowdots)

In [ ]:
! pip install spacy
! python -m spacy download en_core_web_md

In [426]:
import spacy
nlp = spacy.load("en_core_web_md")

In [427]:
sentence = ["The", "dog", "chased", "the", "cat."]
vectors = np.array([nlp(word).vector for word in sentence])

In [ ]:
# Lets look at the vectors
len(vectors), vectors[0].shape

In [429]:
# Understanding dot products
Q = np.array(vectors)
K = np.copy(Q) # copy Queries

In [ ]:
Q.shape, K.shape

In [ ]:
np.dot(Q, K.T)

In [432]:
# embedding size
d_k = len(vectors[0])

# Turn sentence into matrix
Q = np.array(vectors)
K = np.copy(Q)

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K):
    d_k = Q.shape[-1]
    dot = np.dot(Q, K.T)
    scores = dot / np.sqrt(d_k) # square root of embedding size
    attn_weights = softmax(scores) # softmax to scale values
    return attn_weights

print(softmax(np.array([2.0, 1.0, 0.1])), sum(softmax(np.array([2.0, 1.0, 0.1]))) == 1) # normalised
attn_weights = scaled_dot_product_attention(Q, K)

In [ ]:
print("Sentence:", sentence)
print("\nAttention Weights:\n", np.round(attn_weights, 2))

import matplotlib.pyplot as plt
import seaborn as sns

sns.heatmap(attn_weights, annot=True, xticklabels=sentence, yticklabels=sentence, cmap="viridis")
plt.title("Attention Weights")
plt.xlabel("Key Words")
plt.ylabel("Query Words")
plt.show()

In [ ]:
! pip install transformers
! pip install huggingface_hub[hf_xet]

In [436]:
from transformers import AutoTokenizer, AutoModel
import torch

sentence = "the dog chased the cat"
tokens = sentence.lower().split()

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.last_hidden_state[0].numpy()[:len(tokens)]


In [ ]:

sns.heatmap(attn_weights, annot=True, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
plt.title("Attention Weights")
plt.xlabel("Key Words")
plt.ylabel("Query Words")
plt.show()

### How does it actually happen?

In [438]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(0, 1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V) # values dotted with attention weights
    return output, attn_weights


In [439]:
embeddings = torch.from_numpy(embeddings)
d_model = embeddings.shape[-1] # 768 for BERT
# Random projection weights
W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_k)

# Project into Q, K, V
Q = embeddings @ W_q  # shape: (n, d_k)
K = embeddings @ W_k
V = embeddings @ W_v

In [ ]:
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print("Attention output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.heatmap(attn_weights.detach().numpy(), xticklabels=tokens, yticklabels=tokens, annot=True, cmap="viridis")
plt.title("Nonsense Attention Weights")
plt.xlabel("Key")
plt.ylabel("Query")
plt.show()
